# Objective Quest DATAQUEST 5.0 — V12 ModernNCA Learned-Geometry Full-Target Ordinal Regression

Faithful implementation of the preregistered V12 contract. Uses the **official TALENT ModernNCA model implementation**, evaluator-aligned unique respondents, frozen V2 folds, Stage R → Stage S (folds 0/2) → full 5-fold only if promoted, and no fallback submission.


In [1]:
# Dependency setup. Kaggle Internet must be enabled.
# Resolve TALENT main once, then install that exact Git commit for this run.
import sys, subprocess, importlib.util, json, re
from pathlib import Path

DEP_SPECS=[
    'category_encoders>=2.6.3,<3',
    'delu>=0.0.23,<1',
    'einops>=0.8,<1',
    'optuna==3.6.1',
    'qhoptim==1.1.0',
]
subprocess.check_call([sys.executable,'-m','pip','install','-q',*DEP_SPECS])

TALENT_REPO='https://github.com/LAMDA-Tabular/TALENT.git'
lsr=subprocess.check_output(['git','ls-remote',TALENT_REPO,'refs/heads/main'],text=True).strip().split()
if len(lsr)<1 or re.fullmatch(r'[0-9a-f]{40}',lsr[0]) is None:
    raise RuntimeError(f'Could not resolve TALENT main to an exact Git commit: {lsr!r}')
TALENT_INSTALL_COMMIT=lsr[0]
subprocess.check_call([
    sys.executable,'-m','pip','install','-q','--no-deps','--force-reinstall',
    f'git+{TALENT_REPO}@{TALENT_INSTALL_COMMIT}'
])
install_record={'repo':TALENT_REPO,'resolved_commit':TALENT_INSTALL_COMMIT,'dependency_specs':DEP_SPECS}
record_path=Path('/kaggle/working/v12_talent_install.json') if Path('/kaggle/working').exists() else Path('./v12_talent_install.json')
record_path.write_text(json.dumps(install_record,indent=2))
print('Dependency setup complete. TALENT exact commit:',TALENT_INSTALL_COMMIT)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.1/380.1 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 2.6 MB/s eta 0:00:00
Dependency setup complete. TALENT exact commit: 1b973adffa4203c3f62c4949957b1ba3efbe60d3


In [2]:
import os,sys,json,math,random,hashlib,zipfile,platform,subprocess,time,traceback,shutil,copy,inspect,importlib.metadata,ast,re
from pathlib import Path
from dataclasses import dataclass,asdict
import numpy as np, pandas as pd, scipy, sklearn, optuna, torch
from scipy.optimize import minimize
from scipy.stats import spearmanr
from sklearn.metrics import cohen_kappa_score,roc_auc_score,mean_squared_error,mean_absolute_error
from packaging.requirements import Requirement
from packaging.version import Version

SEED=42; HPO_SEED=20260908
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
OUT=Path('/kaggle/working/oqdq5_v12') if Path('/kaggle/working').exists() else Path('./oqdq5_v12')
if OUT.exists(): shutil.rmtree(OUT)
OUT.mkdir(parents=True,exist_ok=True)

@dataclass(frozen=True)
class Cfg:
    target:str='life_satisfaction'; id_col:str='id'; household_col:str='household_id'; n_folds:int=5
    expected_train_rows:int=95609; expected_train_ids:int=94663; expected_test_rows:int=23902; expected_test_ids:int=23666
    screen_folds:tuple=(0,2); train_seeds:tuple=(42,2026); batch_size:int=1024; max_epoch:int=200; patience:int=20
    infer_batch:int=256; bootstrap_reps:int=2000; recovery_lb:float=0.48334
    stage_auc_gain:float=0.005; stage_catastrophic_auc:float=-0.005; stage_qwk_gain:float=0.002
    stage_novel_qwk_floor:float=-0.0015; stage_novel_resid_corr:float=0.95; stage_novel_auc_floor:float=-0.001
    scientific_qwk_gain:float=0.002; submission_qwk_gain:float=0.0015; submission_boot_p:float=0.90
    submission_min_fold:float=-0.0015; challenge_qwk_gain:float=0.0025; challenge_auc_gain:float=0.005
CFG=Cfg()
assert torch.cuda.is_available() and torch.cuda.device_count()>=2, f'T4x2 required; CUDA devices={torch.cuda.device_count()}'
for i in range(2):
    p=torch.cuda.get_device_properties(i); print(i,p.name,round(p.total_memory/2**30,2),'GiB',f'cc={p.major}.{p.minor}')
print('OUT',OUT,'torch',torch.__version__,'numpy',np.__version__,'sklearn',sklearn.__version__,'optuna',optuna.__version__)

0 Tesla T4 14.56 GiB cc=7.5
1 Tesla T4 14.56 GiB cc=7.5
OUT /kaggle/working/oqdq5_v12 torch 2.10.0+cu128 numpy 2.0.2 sklearn 1.6.1 optuna 3.6.1


## 1. Official ModernNCA provenance and frozen HPO contract

In [3]:
import TALENT
from TALENT.model.models.modernNCA import ModernNCA
from TALENT.model.methods.modernNCA import ModernNCAMethod
import importlib.resources as pkg_resources

# Provenance: dependency cell resolved `main` once and installed that exact commit.
_install_record_path=Path('/kaggle/working/v12_talent_install.json') if Path('/kaggle/working').exists() else Path('./v12_talent_install.json')
if not _install_record_path.exists():
    raise RuntimeError('Missing TALENT install provenance record from dependency cell.')
INSTALL_RECORD=json.loads(_install_record_path.read_text())
EXPECTED_INSTALL_COMMIT=INSTALL_RECORD['resolved_commit']
if re.fullmatch(r'[0-9a-f]{40}',EXPECTED_INSTALL_COMMIT) is None:
    raise RuntimeError(f'Invalid resolved TALENT commit: {EXPECTED_INSTALL_COMMIT!r}')

def resolved_git_commit():
    d=importlib.metadata.distribution('TALENT')
    txt=d.read_text('direct_url.json')
    if not txt:
        raise RuntimeError('TALENT direct_url.json is absent; cannot prove Git source provenance.')
    obj=json.loads(txt)
    commit=obj.get('vcs_info',{}).get('commit_id')
    if re.fullmatch(r'[0-9a-f]{40}',str(commit or '')) is None:
        raise RuntimeError(f'TALENT install does not expose a valid Git commit: {obj}')
    return commit,obj

TALENT_COMMIT,DIRECT_URL=resolved_git_commit()
if TALENT_COMMIT != EXPECTED_INSTALL_COMMIT:
    raise RuntimeError(f'TALENT commit mismatch: dependency cell resolved {EXPECTED_INSTALL_COMMIT}, installed package reports {TALENT_COMMIT}.')

# Verify all declared TALENT requirements against the installed Kaggle environment.
req_report=[]
for raw in importlib.metadata.requires('TALENT') or []:
    req=Requirement(raw)
    if req.marker is not None and not req.marker.evaluate():
        continue
    try:
        ver=importlib.metadata.version(req.name)
    except importlib.metadata.PackageNotFoundError as e:
        raise RuntimeError(f'Missing TALENT dependency {req.name} required by {raw}') from e
    ok=(not req.specifier) or (Version(ver) in req.specifier)
    req_report.append({'requirement':raw,'installed':ver,'ok':bool(ok)})
    if not ok:
        raise RuntimeError(f'TALENT dependency mismatch: {raw}, installed {ver}')

model_file=Path(inspect.getfile(ModernNCA)).resolve()
method_file=Path(inspect.getfile(ModernNCAMethod)).resolve()
actual_sha=hashlib.sha256(model_file.read_bytes()).hexdigest()
method_sha=hashlib.sha256(method_file.read_bytes()).hexdigest()
# Hash observed in the failed run is retained as audit context only. Correctness is gated
# by exact commit provenance plus API/config/behavior checks below, not comments/formatting bytes.
LAST_AUDITED_MODEL_SHA='02fce6a107998ab7212774507998e77535f630fbf4ee328acf8519ad7c10632f'

with pkg_resources.files(TALENT).joinpath('configs/default/modernNCA.json').open('r') as f:
    official_default=json.load(f)
with pkg_resources.files(TALENT).joinpath('configs/opt_space/modernNCA.json').open('r') as f:
    official_space=json.load(f)

EXPECTED_DEFAULT={'modernNCA':{'model':{'dim':128,'dropout':0.1,'d_block':512,'n_blocks':0,'temperature':1,'num_embeddings':{'type':'PLREmbeddings','n_frequencies':77,'frequency_scale':0.04431360576139521,'d_embedding':34,'lite':True},'sample_rate':0.5},'training':{'lr':0.01,'weight_decay':0.0002},'general':{}}}
EXPECTED_SPACE={'modernNCA':{'model':{'dropout':['uniform',0.0,0.5],'d_block':['int',64,1024],'n_blocks':['?int',0,0,2],'dim':['int',64,1024],'num_embeddings':{'n_frequencies':['int',16,96],'frequency_scale':['loguniform',0.005,10.0],'d_embedding':['int',16,64]},'sample_rate':['uniform',0.05,0.6]},'training':{'lr':['loguniform',1e-05,0.1],'weight_decay':['?loguniform',0.0,1e-06,0.001]},'general':{}}}
if official_default != EXPECTED_DEFAULT:
    raise RuntimeError('Official ModernNCA default config changed; V12 contract requires re-audit.')
if official_space != EXPECTED_SPACE:
    raise RuntimeError('Official ModernNCA HPO space changed; V12 contract requires re-audit.')

# API + behavioral integrity checks. These guard the actual scientific mechanism used by V12.
ctor=list(inspect.signature(ModernNCA.__init__).parameters)
expected_ctor=['self','d_in','d_num','d_out','dim','dropout','d_block','n_blocks','num_embeddings','temperature','sample_rate']
if ctor != expected_ctor:
    raise RuntimeError(f'ModernNCA constructor drift: {ctor}')
fwd=list(inspect.signature(ModernNCA.forward).parameters)
if fwd != ['self','x','y','candidate_x','candidate_y','is_train']:
    raise RuntimeError(f'ModernNCA forward signature drift: {fwd}')
src=inspect.getsource(ModernNCA.forward)
for token in [
    'retrival_size = int(data_size * self.sample_rate)',
    'torch.randperm(data_size)',
    'candidate_x = candidate_x[sample_idx]',
    'torch.cdist(x, candidate_x, p=2)',
    'fill_diagonal_(torch.inf)',
    'F.softmax(-distances',
    'torch.mm(distances, candidate_y)',
]:
    if token not in src:
        raise RuntimeError(f'ModernNCA behavior drift: missing {token!r}')
init_src=inspect.getsource(ModernNCAMethod.__init__)
for token in ["cat_policy == 'tabr_ohe'","num_policy == 'none'"]:
    if token not in init_src:
        raise RuntimeError(f'ModernNCA method preprocessing contract drift: missing {token!r}')
train_src=inspect.getsource(ModernNCAMethod.train_epoch)
for token in ['make_random_batches','candidate_indices','torch.isin','candidate_y','is_train = True','loss.backward()','self.optimizer.step()']:
    if token not in train_src:
        raise RuntimeError(f'ModernNCA training semantics drift: missing {token!r}')
predict_src=inspect.getsource(ModernNCAMethod.predict)
for token in ['candidate_x','candidate_y','is_train = False',"y_info.get('policy') == 'mean_std'"]:
    if token not in predict_src:
        raise RuntimeError(f'ModernNCA prediction semantics drift: missing {token!r}')
if tuple(map(int,torch.__version__.split('+')[0].split('.')[:2])) < (2,8):
    raise RuntimeError(f'TALENT requires torch>=2.8, found {torch.__version__}')

# The preregistered eight configurations are frozen explicitly so TALENT/Optuna sampling
# utility drift cannot alter the experiment. They are the exact official-space draws from HPO_SEED.
CONFIGS=[{'model': {'dim': 422, 'dropout': 0.29961907325969833, 'd_block': 269, 'n_blocks': 0, 'temperature': 1, 'num_embeddings': {'type': 'PLREmbeddings', 'n_frequencies': 35, 'frequency_scale': 0.2808646588975522, 'd_embedding': 31, 'lite': True}, 'sample_rate': 0.31765543760789716}, 'training': {'lr': 0.01763072419056641, 'weight_decay': 0.0}, 'general': {}, 'config_id': 0}, {'model': {'dim': 781, 'dropout': 0.38221205108011186, 'd_block': 706, 'n_blocks': 0, 'temperature': 1, 'num_embeddings': {'type': 'PLREmbeddings', 'n_frequencies': 35, 'frequency_scale': 0.007849896600161546, 'd_embedding': 32, 'lite': True}, 'sample_rate': 0.3057587504534756}, 'training': {'lr': 0.0006054262918055389, 'weight_decay': 0.0}, 'general': {}, 'config_id': 1}, {'model': {'dim': 177, 'dropout': 0.332544262105686, 'd_block': 304, 'n_blocks': 0, 'temperature': 1, 'num_embeddings': {'type': 'PLREmbeddings', 'n_frequencies': 60, 'frequency_scale': 8.23268524394698, 'd_embedding': 32, 'lite': True}, 'sample_rate': 0.4409223428658866}, 'training': {'lr': 0.0008655326388031571, 'weight_decay': 2.6841984147952544e-05}, 'general': {}, 'config_id': 2}, {'model': {'dim': 852, 'dropout': 0.4742625722425824, 'd_block': 686, 'n_blocks': 0, 'temperature': 1, 'num_embeddings': {'type': 'PLREmbeddings', 'n_frequencies': 52, 'frequency_scale': 0.014451539830596159, 'd_embedding': 23, 'lite': True}, 'sample_rate': 0.5414846456561201}, 'training': {'lr': 0.03350566263168187, 'weight_decay': 4.0818352891187185e-06}, 'general': {}, 'config_id': 3}, {'model': {'dim': 1001, 'dropout': 0.0824739908891644, 'd_block': 244, 'n_blocks': 0, 'temperature': 1, 'num_embeddings': {'type': 'PLREmbeddings', 'n_frequencies': 44, 'frequency_scale': 1.1475088799828153, 'd_embedding': 36, 'lite': True}, 'sample_rate': 0.06325492185297743}, 'training': {'lr': 0.004112476281724047, 'weight_decay': 0.00010371067653521566}, 'general': {}, 'config_id': 4}, {'model': {'dim': 509, 'dropout': 0.27009069276603764, 'd_block': 415, 'n_blocks': 0, 'temperature': 1, 'num_embeddings': {'type': 'PLREmbeddings', 'n_frequencies': 29, 'frequency_scale': 0.36879193722958054, 'd_embedding': 58, 'lite': True}, 'sample_rate': 0.5710657698916356}, 'training': {'lr': 0.0018452700480821468, 'weight_decay': 0.00014749005154614023}, 'general': {}, 'config_id': 5}, {'model': {'dim': 517, 'dropout': 0.341523178057644, 'd_block': 389, 'n_blocks': 0, 'temperature': 1, 'num_embeddings': {'type': 'PLREmbeddings', 'n_frequencies': 82, 'frequency_scale': 6.838958867225874, 'd_embedding': 56, 'lite': True}, 'sample_rate': 0.18707197172477863}, 'training': {'lr': 5.751581692675271e-05, 'weight_decay': 0.00015963944524800282}, 'general': {}, 'config_id': 6}, {'model': {'dim': 402, 'dropout': 0.378830499948205, 'd_block': 808, 'n_blocks': 0, 'temperature': 1, 'num_embeddings': {'type': 'PLREmbeddings', 'n_frequencies': 51, 'frequency_scale': 3.776827150217538, 'd_embedding': 22, 'lite': True}, 'sample_rate': 0.21656174408896478}, 'training': {'lr': 0.000984120733887293, 'weight_decay': 0.0}, 'general': {}, 'config_id': 7}]
CONFIG_MANIFEST_EXPECTED_SHA='ab65458a3176ab4c48f706fee2ff93a9a927b4eab3ad9fcdb64d3eb410ced72d'
config_sha=hashlib.sha256(json.dumps(CONFIGS,sort_keys=True).encode()).hexdigest()
if config_sha != CONFIG_MANIFEST_EXPECTED_SHA:
    raise RuntimeError(f'Frozen V12 config manifest corrupted: {config_sha}')
if [c['config_id'] for c in CONFIGS] != list(range(8)):
    raise RuntimeError('Frozen ModernNCA config IDs must be exactly 0..7.')

# Quality-agnostic validation that every frozen draw remains inside the exact official search space.
def _in_range(name,val,spec):
    kind=spec[0].lstrip('?')
    if kind=='int':
        return isinstance(val,(int,np.integer)) and not isinstance(val,bool) and spec[-2] <= int(val) <= spec[-1]
    if kind in ('uniform','loguniform'):
        return isinstance(val,(int,float,np.integer,np.floating)) and spec[-2] <= float(val) <= spec[-1]
    return False
for c in CONFIGS:
    m,t=c['model'],c['training']
    for k in ('dropout','d_block','dim','sample_rate'):
        if not _in_range(k,m[k],EXPECTED_SPACE['modernNCA']['model'][k]):
            raise RuntimeError(f'Frozen config {c["config_id"]} {k} outside official space.')
    # Optional n_blocks/weight_decay may legitimately equal their official default.
    if m['n_blocks'] != 0 and not _in_range('n_blocks',m['n_blocks'],EXPECTED_SPACE['modernNCA']['model']['n_blocks']):
        raise RuntimeError(f'Frozen config {c["config_id"]} n_blocks outside official space.')
    ne=m['num_embeddings']
    for k in ('n_frequencies','frequency_scale','d_embedding'):
        if not _in_range(k,ne[k],EXPECTED_SPACE['modernNCA']['model']['num_embeddings'][k]):
            raise RuntimeError(f'Frozen config {c["config_id"]} num_embeddings.{k} outside official space.')
    if not _in_range('lr',t['lr'],EXPECTED_SPACE['modernNCA']['training']['lr']):
        raise RuntimeError(f'Frozen config {c["config_id"]} lr outside official space.')
    if t['weight_decay'] != 0.0 and not _in_range('weight_decay',t['weight_decay'],EXPECTED_SPACE['modernNCA']['training']['weight_decay']):
        raise RuntimeError(f'Frozen config {c["config_id"]} weight_decay outside official space.')

with open(OUT/'modernnca_configs_v12.json','w') as f:
    json.dump(CONFIGS,f,indent=2)

source_contract={
    'TALENT_commit':TALENT_COMMIT,
    'modernNCA_model_sha256':actual_sha,
    'modernNCA_method_sha256':method_sha,
    'model_sha_matches_failed_run_audit':actual_sha==LAST_AUDITED_MODEL_SHA,
    'config_manifest_sha256':config_sha,
    'dependency_report':req_report,
    'constructor':ctor,
    'forward_signature':fwd,
}
with open(OUT/'talent_source_contract_v12.json','w') as f:
    json.dump(source_contract,f,indent=2)
print('TALENT',getattr(TALENT,'__version__','unknown'),'commit',TALENT_COMMIT,'model_sha',actual_sha,'config_sha',config_sha)
display(pd.DataFrame([{'config_id':c['config_id'],**{k:v for k,v in c['model'].items() if k!='num_embeddings'},**{f'num_{k}':v for k,v in c['model']['num_embeddings'].items() if k not in ('type','lite')},**c['training']} for c in CONFIGS]))


TALENT 0.1.0 commit 1b973adffa4203c3f62c4949957b1ba3efbe60d3 model_sha 02fce6a107998ab7212774507998e77535f630fbf4ee328acf8519ad7c10632f config_sha ab65458a3176ab4c48f706fee2ff93a9a927b4eab3ad9fcdb64d3eb410ced72d


,config_id,dim,dropout,d_block,n_blocks,temperature,sample_rate,num_n_frequencies,num_frequency_scale,num_d_embedding,lr,weight_decay
0,0,422,0.299619,269,0,1,0.317655,35,0.280865,31,0.017631,0.000000
1,1,781,0.382212,706,0,1,0.305759,35,0.007850,32,0.000605,0.000000
2,2,177,0.332544,304,0,1,0.440922,60,8.232685,32,0.000866,0.000027
3,3,852,0.474263,686,0,1,0.541485,52,0.014452,23,0.033506,0.000004
4,4,1001,0.082474,244,0,1,0.063255,44,1.147509,36,0.004112,0.000104
5,5,509,0.270091,415,0,1,0.571066,29,0.368792,58,0.001845,0.000147
6,6,517,0.341523,389,0,1,0.187072,82,6.838959,56,0.000058,0.000160
7,7,402,0.378830,808,0,1,0.216562,51,3.776827,22,0.000984,0.000000


## 2. Competition data, historical control, frozen folds, semantic representation

In [4]:
def roots():
    r=[Path('/kaggle/input'),Path('/mnt/data')]
    c=Path.cwd().resolve()
    if c!=Path('/') and not str(c).startswith('/proc'): r.append(c)
    return [x for x in r if x.exists()]

def locate_data():
    hits=[]
    for root in roots():
        for p in root.rglob('train_final.csv'):
            d=p.parent
            if (d/'test_final.csv').exists() and (d/'data_dictionary.csv').exists(): hits.append(d.resolve())
    hits=list(dict.fromkeys(hits));
    if not hits: raise FileNotFoundError('Competition train/test/data_dictionary not found.')
    hits.sort(key=lambda p:(not (p/'sample_submission.csv').exists(),len(str(p))))
    return hits[0]

class Store:
    def __init__(self,root=None,z=None): self.root=Path(root) if root else None; self.z=Path(z) if z else None; self.zf=zipfile.ZipFile(self.z) if self.z else None
    def exists(self,rel): return (self.root/rel).exists() if self.root else rel in self.zf.namelist()
    def csv(self,rel,**kw):
        if self.root: return pd.read_csv(self.root/rel,**kw)
        with self.zf.open(rel) as f: return pd.read_csv(f,**kw)
    def source(self): return str(self.root or self.z)
    def close(self):
        if self.zf: self.zf.close()

def locate_store():
    marker='results/v2/oof_v2.csv'; dirs=[]; zs=[]
    for root in roots():
        for p in root.rglob('oof_v2.csv'):
            if p.as_posix().endswith('/results/v2/oof_v2.csv'): dirs.append(p.parents[2].resolve())
        for z in root.rglob('*.zip'):
            try:
                with zipfile.ZipFile(z) as q:
                    if marker in q.namelist(): zs.append(z.resolve())
            except zipfile.BadZipFile: pass
    dirs=list(dict.fromkeys(dirs)); zs=list(dict.fromkeys(zs))
    if dirs: return Store(root=sorted(dirs,key=lambda p:len(str(p)))[0])
    if len(zs)==1: return Store(z=zs[0])
    if not zs: raise FileNotFoundError('Upload completed historical results.zip as Kaggle input.')
    # prefer a file literally named results.zip if multiple archives coexist
    named=[z for z in zs if z.name=='results.zip']
    if len(named)==1: return Store(z=named[0])
    raise RuntimeError(f'Ambiguous historical archives: {zs}')

DATA=locate_data(); STORE=locate_store()
train=pd.read_csv(DATA/'train_final.csv'); test=pd.read_csv(DATA/'test_final.csv'); dd=pd.read_csv(DATA/'data_dictionary.csv')
assert len(train)==CFG.expected_train_rows and train.id.nunique()==CFG.expected_train_ids
assert len(test)==CFG.expected_test_rows and test.id.nunique()==CFG.expected_test_ids
# Duplicated respondent rows must be feature-identical and target-consistent.
for df,name,has_y in [(train,'train',True),(test,'test',False)]:
    dup=df[df.duplicated('id',False)]
    if len(dup):
        g=dup.groupby('id',sort=False)
        cols=[c for c in df.columns if c!='id']
        bad=[]
        for c in cols:
            if g[c].nunique(dropna=False).max()>1: bad.append(c)
        assert not bad, f'{name} duplicate IDs differ in columns: {bad[:10]}'

v2=STORE.csv('results/v2/oof_v2.csv',usecols=['id','life_satisfaction','primary_fold'])
canon=v2.drop_duplicates('id',keep='first').reset_index(drop=True)
assert len(canon)==CFG.expected_train_ids and set(canon.primary_fold)==set(range(5))
train_uid=canon[['id']].merge(train.drop_duplicates('id',keep='first'),on='id',validate='one_to_one')
assert np.array_equal(train_uid.life_satisfaction.astype(int),canon.life_satisfaction.astype(int))
train_uid['__fold__']=canon.primary_fold.astype(int).to_numpy()
test_uid=test.drop_duplicates('id',keep='first').reset_index(drop=True)
assert set(train_uid.id).isdisjoint(set(test_uid.id))

FEATURES=[c for c in test.columns if c not in {CFG.id_col,CFG.household_col}]
CONT=[c for c in ['income_to_poverty_ratio','hours_worked_per_week','survey_weight','days_missed_work','height_inches','age','weight_pounds'] if c in FEATURES]
ORD=[c for c in ['education_level','max_education_in_family','income_poverty_category','food_security_4cat','food_security_3cat','general_health_status','depression_frequency','anxiety_frequency','anxiety_level','survey_year','data_year'] if c in FEATURES]
NOM=[c for c in FEATURES if c not in set(CONT+ORD)]
SPECIAL={'weight_pounds':[996,997,999],'height_inches':[96,97,98,99],'hours_worked_per_week':[990],'days_missed_work':[997,998,999],'income_to_poverty_ratio':[110]}
SPECIAL={k:v for k,v in SPECIAL.items() if k in CONT}
TYPEMAP={c:('continuous' if c in CONT else 'ordinal' if c in ORD else 'nominal') for c in FEATURES}
assert CFG.target not in FEATURES and CFG.household_col not in FEATURES and CFG.id_col not in FEATURES

train_uid.to_csv(OUT/'v12_train_unique_raw.csv',index=False); test_uid.to_csv(OUT/'v12_test_unique_raw.csv',index=False)
with open(OUT/'semantic_contract_v12.json','w') as f: json.dump({'features':FEATURES,'continuous':CONT,'ordinal':ORD,'nominal':NOM,'special_codes':SPECIAL,'typing':'dictionary/audited semantics only; target-independent'},f,indent=2)
print('DATA',DATA,'ARTIFACTS',STORE.source(),'unique',len(train_uid),len(test_uid),'types',len(CONT),len(ORD),len(NOM))

DATA /kaggle/input/competitions/oq-dq-5 ARTIFACTS /kaggle/input/datasets/riantamaputra/dataquest-2026-historical-results unique 94663 23666 types 7 11 158


In [5]:
# Frozen recovery champion reconstruction (same exact recipe that produced Public LB 0.48334).
COMP={
'v2_final':('v2','oof_v2.csv','test_predictions_v2.csv','oof_final_continuous','test_final_continuous'),
'v5_final':('v5_1','oof_v5.csv','test_predictions_v5.csv','oof_final_honest_continuous','pred_final_continuous'),
'v61_final':('v6_1','oof_v6.csv','test_predictions_v6.csv','oof_final_crossfit','pred_final_continuous'),
'v7_stack':('v7','oof_v7.csv','test_scores_v7.csv','v7_stack','v7_stack')}
META={'v2':('primary_fold','life_satisfaction'),'v5_1':('fold','life_satisfaction'),'v6_1':('primary_fold','life_satisfaction'),'v7':('primary_fold','life_satisfaction')}
test_raw=test.copy(); test_ids=test_uid.id.to_numpy(); train_ids=train_uid.id.to_numpy(); y=train_uid.life_satisfaction.astype(int).to_numpy(); folds=train_uid.__fold__.astype(int).to_numpy()

def test_ids_for(df):
    if 'id' in df: return df.id
    if 'test_feature_id' in df: return df.test_feature_id
    if 'row_index' in df:
        assert np.array_equal(df.row_index.to_numpy(),np.arange(len(test_raw))); return test_raw.id.reset_index(drop=True)
    raise KeyError('No test ID mapping')

def load_component(name,spec):
    v,of,tf,oc,tc=spec; od=STORE.csv(f'results/{v}/{of}'); td=STORE.csv(f'results/{v}/{tf}')
    fc,yc=META[v]
    g=od[['id',yc,fc,oc]].groupby('id',sort=False)
    assert g[yc].nunique().max()==1 and g[fc].nunique().max()==1 and (g[oc].max()-g[oc].min()).max()<1e-8
    a=g.agg({yc:'first',oc:'mean'}).reset_index(); a=canon[['id']].merge(a,on='id',validate='one_to_one')
    assert np.array_equal(a[yc].astype(int),y)
    td=td.copy(); td['_id']=test_ids_for(td).to_numpy(); h=td[['_id',tc]].groupby('_id',sort=False)
    assert (h[tc].max()-h[tc].min()).max()<1e-8
    b=h[tc].mean().reset_index().rename(columns={'_id':'id'}); b=test_uid[['id']].merge(b,on='id',validate='one_to_one')
    assert a[oc].notna().all() and b[tc].notna().all()
    return a[oc].to_numpy(float),b[tc].to_numpy(float)
arr={n:load_component(n,s) for n,s in COMP.items()}
REC_NAMES=('v2_final','v5_final','v61_final','v7_stack')
RW={0:(0.6131481145225551,0.1984633896432871,0.18838849583415773,0.0),1:(0.5736087705637004,0.2130793852900037,0.20117728057551384,0.012134563570782105),2:(0.5260926035320435,0.22651646669428574,0.20045779567769734,0.04693313409597358),3:(0.412997730762594,0.12147998345945334,0.2995554634343377,0.16596682234361498),4:(0.4419367210113302,0.27031416820745807,0.2416404622741877,0.04610864850702414)}
RT={0:(1.5694444444444446,2.011875,2.4941666666666666),1:(1.57,2.011875,2.5100000000000002),2:(1.57,2.0162500000000003,2.5100000000000002),3:(1.5683333333333334,2.011875,2.524166666666667),4:(1.568888888888889,2.014375,2.5100000000000002)}
RFW=np.array((0.5333147962860486,0.2160045364556684,0.20393903979487613,0.0467416274634069)); RFT=np.array((1.5694444444444446,2.011875,2.5100000000000002))
RX=np.column_stack([arr[n][0] for n in REC_NAMES]); RTX=np.column_stack([arr[n][1] for n in REC_NAMES])
recovery_oof=np.empty(len(y)); recovery_class=np.empty(len(y),int)
for f in range(5):
    m=folds==f; recovery_oof[m]=RX[m]@np.array(RW[f]); t=np.array(RT[f]); recovery_class[m]=1+(recovery_oof[m]>=t[0])+(recovery_oof[m]>=t[1])+(recovery_oof[m]>=t[2])
recovery_test=RTX@RFW; recovery_test_class=1+(recovery_test>=RFT[0])+(recovery_test>=RFT[1])+(recovery_test>=RFT[2])
v2_oof=arr['v2_final'][0].copy(); v2_test=arr['v2_final'][1].copy()
rec_qwk=cohen_kappa_score(y,recovery_class,weights='quadratic'); assert abs(rec_qwk-0.48186721909562624)<1e-12
np.savez_compressed(OUT/'recovery_control_v12.npz',id=train_ids,y=y,fold=folds,oof=recovery_oof,cls=recovery_class,test_id=test_ids,test_score=recovery_test,test_cls=recovery_test_class,v2_oof=arr['v2_final'][0],v2_test=arr['v2_final'][1])
print('Frozen recovery OOF',rec_qwk,'Public LB',CFG.recovery_lb)

Frozen recovery OOF 0.48186721909562624 Public LB 0.48334


## 3. Isolated official-ModernNCA worker

One worker sees exactly one GPU. Training uses the official `ModernNCA` class and its native stochastic neighbor sampling / soft-neighbor regression. The custom loop only provides nested split control, fixed-epoch outer refits, inference chunking, and artifact telemetry.

In [6]:
WORKER_SOURCE = '#!/usr/bin/env python\nimport os,sys,json,time,gc,traceback,argparse,random,hashlib,inspect,re,importlib.metadata\nfrom pathlib import Path\nos.environ.setdefault(\'PYTORCH_CUDA_ALLOC_CONF\',\'expandable_segments:True\')\nos.environ.setdefault(\'CUBLAS_WORKSPACE_CONFIG\',\':4096:8\')\nimport numpy as np,pandas as pd,torch,joblib\nfrom sklearn.preprocessing import OneHotEncoder,StandardScaler\nfrom sklearn.model_selection import StratifiedShuffleSplit\nfrom sklearn.metrics import mean_squared_error\nfrom TALENT.model.models.modernNCA import ModernNCA\n\nP=argparse.ArgumentParser(); P.add_argument(\'--root\',required=True); P.add_argument(\'--mode\',choices=[\'runtime\',\'screen\',\'full\',\'resume_test\'],required=True); P.add_argument(\'--fold\',type=int,required=True); a=P.parse_args()\nROOT=Path(a.root); MODE=a.mode; FOLD=a.fold; DEVICE=torch.device(\'cuda:0\')\nif not torch.cuda.is_available() or torch.cuda.device_count()!=1: raise RuntimeError(f\'Worker must see exactly one GPU, got {torch.cuda.device_count()}\')\nCFG=json.load(open(ROOT/\'runtime_contract.json\')); CONFIGS=json.load(open(ROOT/\'modernnca_configs_v12.json\')); SEM=json.load(open(ROOT/\'semantic_contract_v12.json\'))\n# Prove child process uses the same official TALENT source/config contract as the parent.\n_model_file=Path(inspect.getfile(ModernNCA)).resolve()\n_child_model_sha=hashlib.sha256(_model_file.read_bytes()).hexdigest()\nif _child_model_sha != CFG[\'modernnca_model_sha256\']:\n    raise RuntimeError(f\'Worker ModernNCA source mismatch: parent={CFG["modernnca_model_sha256"]}, child={_child_model_sha}\')\ntry:\n    _du=json.loads(importlib.metadata.distribution(\'TALENT\').read_text(\'direct_url.json\') or \'{}\')\n    _child_commit=_du.get(\'vcs_info\',{}).get(\'commit_id\')\nexcept Exception as e:\n    raise RuntimeError(\'Worker cannot resolve TALENT Git provenance\') from e\nif _child_commit != CFG[\'talent_commit\']:\n    raise RuntimeError(f\'Worker TALENT commit mismatch: parent={CFG["talent_commit"]}, child={_child_commit}\')\n_child_cfg_sha=hashlib.sha256(json.dumps(CONFIGS,sort_keys=True).encode()).hexdigest()\nif _child_cfg_sha != CFG[\'config_manifest_sha256\'] or len(CONFIGS)!=8 or [c[\'config_id\'] for c in CONFIGS]!=list(range(8)):\n    raise RuntimeError(f\'Worker fixed-config contract mismatch: {_child_cfg_sha}\')\n_ctor=list(inspect.signature(ModernNCA.__init__).parameters)\n_fwd_src=inspect.getsource(ModernNCA.forward)\nif _ctor != [\'self\',\'d_in\',\'d_num\',\'d_out\',\'dim\',\'dropout\',\'d_block\',\'n_blocks\',\'num_embeddings\',\'temperature\',\'sample_rate\']:\n    raise RuntimeError(f\'Worker ModernNCA constructor drift: {_ctor}\')\nfor _token in [\'self.sample_rate\',\'torch.cdist\',\'fill_diagonal_\',\'F.softmax\',\'torch.mm\']:\n    if _token not in _fwd_src: raise RuntimeError(f\'Worker ModernNCA behavior drift: missing {_token}\')\nTRAIN=pd.read_csv(ROOT/\'v12_train_unique_raw.csv\'); TEST=pd.read_csv(ROOT/\'v12_test_unique_raw.csv\')\nSEED=42; BATCH=int(CFG[\'batch_size\']); INFER=int(CFG[\'infer_batch\']); MAXE=int(CFG[\'max_epoch\']); PAT=int(CFG[\'patience\'])\nrandom.seed(SEED+FOLD); np.random.seed(SEED+FOLD); torch.manual_seed(SEED+FOLD); torch.cuda.manual_seed_all(SEED+FOLD)\nOUT=ROOT/f\'{MODE}_fold{FOLD}\'; OUT.mkdir(parents=True,exist_ok=True)\n\ndef mem(tag):\n    torch.cuda.synchronize(); d={\'tag\':tag,\'allocated\':int(torch.cuda.memory_allocated()),\'reserved\':int(torch.cuda.memory_reserved()),\'peak\':int(torch.cuda.max_memory_allocated())}; print(\'MEM\',json.dumps(d),flush=True); return d\n\nclass Prep:\n    def __init__(self): self.cont=SEM[\'continuous\']; self.ord=SEM[\'ordinal\']; self.nom=SEM[\'nominal\']; self.special={k:set(map(float,v)) for k,v in SEM[\'special_codes\'].items()}\n    def fit(self,df):\n        self.cont_med={}; self.ord_med={}\n        C=[]\n        for c in self.cont:\n            s=pd.to_numeric(df[c],errors=\'coerce\').astype(float); s=s.mask(s.isin(self.special.get(c,set())))\n            self.cont_med[c]=float(s.median())\n            C.append(s.fillna(self.cont_med[c]).to_numpy(float))\n        self.cscale=StandardScaler().fit(np.column_stack(C)) if C else None\n        for c in self.ord:\n            s=pd.to_numeric(df[c],errors=\'coerce\').astype(float); self.ord_med[c]=float(s.median())\n        raw_missing=df[SEM[\'features\']].isna().sum(axis=1).to_numpy(float)\n        sp=np.zeros(len(df))\n        for c,v in self.special.items(): sp += pd.to_numeric(df[c],errors=\'coerce\').astype(float).isin(v).to_numpy(float)\n        self.miss_scale=StandardScaler().fit((raw_missing+sp).reshape(-1,1))\n        self.ohe=OneHotEncoder(handle_unknown=\'ignore\',sparse_output=False,dtype=np.float32)\n        self.ohe.fit(df[self.nom].astype(object).where(df[self.nom].notna(),\'__MISSING__\').astype(str))\n        self.n_num=len(self.cont)+len(self.ord)+1\n        return self\n    def transform(self,df):\n        nums=[]; flags=[]\n        if self.cont:\n            base=[]\n            for c in self.cont:\n                orig=pd.to_numeric(df[c],errors=\'coerce\').astype(float); sf=orig.isin(self.special.get(c,set())); miss=orig.isna()|sf\n                base.append(orig.mask(sf).fillna(self.cont_med[c]).to_numpy(float)); flags.append(miss.to_numpy(np.float32));\n                if c in self.special: flags.append(sf.to_numpy(np.float32))\n            nums.extend(self.cscale.transform(np.column_stack(base)).T)\n        for c in self.ord: nums.append(pd.to_numeric(df[c],errors=\'coerce\').astype(float).fillna(self.ord_med[c]).to_numpy(float))\n        raw_missing=df[SEM[\'features\']].isna().sum(axis=1).to_numpy(float); sp=np.zeros(len(df))\n        for c,v in self.special.items(): sp += pd.to_numeric(df[c],errors=\'coerce\').astype(float).isin(v).to_numpy(float)\n        nums.append(self.miss_scale.transform((raw_missing+sp).reshape(-1,1)).ravel())\n        N=np.column_stack(nums).astype(np.float32)\n        C=self.ohe.transform(df[self.nom].astype(object).where(df[self.nom].notna(),\'__MISSING__\').astype(str)).astype(np.float32)\n        if flags: C=np.column_stack([C]+flags).astype(np.float32)\n        X=np.ascontiguousarray(np.column_stack([N,C]).astype(np.float32))\n        assert np.isfinite(X).all()\n        return X\n    def manifest(self): return {\'n_num\':self.n_num,\'ohe_dim\':int(sum(map(len,self.ohe.categories_))),\'total_nominal_categories\':{c:int(len(v)) for c,v in zip(self.nom,self.ohe.categories_)}}\n\ndef build_model(d_in,d_num,cfg):\n    mc={k:v for k,v in cfg[\'model\'].items()}\n    return ModernNCA(d_in=d_in,d_num=d_num,d_out=1,**mc).to(DEVICE).float()\n\ndef _encode_eval(model,x):\n    if model.num_embeddings is not None and model.d_num>0:\n        xn,xc=x[:,:model.d_num],x[:,model.d_num:]\n        xn=model.num_embeddings(xn).flatten(1)\n        x=torch.cat([xn,xc],dim=-1)\n    x=model.encoder(x)\n    if model.n_blocks>0: x=model.post_encoder(x)\n    return x\n\ndef _encode_support(model,Xs,chunk=2048):\n    zs=[]\n    with torch.no_grad():\n        for s in range(0,len(Xs),chunk):\n            zs.append(_encode_eval(model,Xs[s:s+chunk]))\n    return torch.cat(zs,dim=0)\n\ndef pred(model,Xq,Xs,ys,mean,std,chunk=INFER):\n    # Exact is_train=False ModernNCA prediction, with support embeddings cached once.\n    # This changes only execution efficiency: same encoder, Euclidean distance, temperature,\n    # softmax weights, and full-support weighted target average as official forward().\n    model.eval(); out=[]\n    with torch.no_grad():\n        Zs=_encode_support(model,Xs)\n        cy=ys.reshape(-1,1)\n        for s in range(0,len(Xq),chunk):\n            q=torch.as_tensor(Xq[s:s+chunk],device=DEVICE,dtype=torch.float32)\n            Zq=_encode_eval(model,q)\n            d=torch.cdist(Zq,Zs,p=2)/model.T\n            weights=torch.softmax(-d,dim=-1)\n            z=torch.mm(weights,cy).reshape(-1)\n            out.append(z.float().cpu().numpy())\n        del Zs\n    ans=np.concatenate(out)*std+mean\n    if not np.isfinite(ans).all():\n        raise FloatingPointError(\'Non-finite ModernNCA predictions\')\n    return ans\n\ndef train_early(Xtr,ytr,Xv,yv,cfg,seed):\n    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()\n    mean=float(np.mean(ytr)); std=float(np.std(ytr)); std=std if std>0 else 1.0\n    yt=(ytr-mean)/std; yvt=(yv-mean)/std\n    X=torch.as_tensor(Xtr,device=DEVICE,dtype=torch.float32); Y=torch.as_tensor(yt,device=DEVICE,dtype=torch.float32)\n    model=build_model(X.shape[1],PREP_NNUM,cfg); opt=torch.optim.AdamW(model.parameters(),lr=float(cfg[\'training\'][\'lr\']),weight_decay=float(cfg[\'training\'][\'weight_decay\']))\n    best=np.inf; best_epoch=-1; best_state=None; stale=0; n=len(Xtr); allidx=torch.arange(n,device=DEVICE)\n    hist=[]\n    for ep in range(MAXE):\n        model.train(); perm=torch.randperm(n,device=DEVICE); losses=[]\n        for bi in perm.split(BATCH):\n            mask=torch.ones(n,dtype=torch.bool,device=DEVICE); mask[bi]=False\n            pr=model(X[bi],Y[bi],X[mask],Y[mask],True).reshape(-1); loss=torch.nn.functional.mse_loss(pr,Y[bi])\n            if not torch.isfinite(loss): raise FloatingPointError(f\'Non-finite HPO train loss epoch={ep}\')\n            opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); losses.append(float(loss.detach()))\n        pv=pred(model,Xv,X,Y,mean,std); rm=float(np.sqrt(mean_squared_error(yv,pv)))\n        if not np.isfinite(rm): raise FloatingPointError(f\'Non-finite validation RMSE epoch={ep}\')\n        hist.append({\'epoch\':ep,\'rmse\':rm,\'train_loss\':float(np.mean(losses))})\n        if rm < best-1e-10:\n            best=rm; best_epoch=ep; best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; stale=0\n        else: stale+=1\n        if stale>PAT: break\n    model.load_state_dict(best_state); return model,best,best_epoch,mean,std,hist,X,Y\n\ndef train_fixed(Xtr,ytr,cfg,seed,epochs):\n    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()\n    mean=float(np.mean(ytr)); std=float(np.std(ytr)); std=std if std>0 else 1.0; yt=(ytr-mean)/std\n    X=torch.as_tensor(Xtr,device=DEVICE,dtype=torch.float32); Y=torch.as_tensor(yt,device=DEVICE,dtype=torch.float32)\n    model=build_model(X.shape[1],PREP_NNUM,cfg); opt=torch.optim.AdamW(model.parameters(),lr=float(cfg[\'training\'][\'lr\']),weight_decay=float(cfg[\'training\'][\'weight_decay\']))\n    n=len(Xtr)\n    for ep in range(int(epochs)):\n        model.train(); perm=torch.randperm(n,device=DEVICE)\n        for bi in perm.split(BATCH):\n            mask=torch.ones(n,dtype=torch.bool,device=DEVICE); mask[bi]=False\n            pr=model(X[bi],Y[bi],X[mask],Y[mask],True).reshape(-1); loss=torch.nn.functional.mse_loss(pr,Y[bi])\n            if not torch.isfinite(loss): raise FloatingPointError(f\'Non-finite refit train loss epoch={ep}\')\n            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()\n    return model,mean,std,X,Y\n\ndef embed(model,X,chunk=1024):\n    out=[]; model.eval()\n    with torch.no_grad():\n        for s in range(0,len(X),chunk):\n            x=torch.as_tensor(X[s:s+chunk],device=DEVICE,dtype=torch.float32)\n            if model.num_embeddings is not None and model.d_num>0:\n                xn,xc=x[:,:model.d_num],x[:,model.d_num:]; xn=model.num_embeddings(xn).flatten(1); x=torch.cat([xn,xc],-1)\n            x=model.encoder(x)\n            if model.n_blocks>0: x=model.post_encoder(x)\n            out.append(x.float().cpu().numpy())\n    return np.concatenate(out)\n\ndef geometry(model,Xv,Xtr,yv,ytr):\n    # Seed-42 model only; diagnostic, never used for selection.\n    Zs=embed(model,Xtr); Zq=embed(model,Xv); rows=[]\n    Zs_t=torch.as_tensor(Zs,device=DEVICE); yt=torch.as_tensor(ytr,device=DEVICE,dtype=torch.float32)\n    for s in range(0,len(Zq),128):\n        q=torch.as_tensor(Zq[s:s+128],device=DEVICE); d=torch.cdist(q,Zs_t); w=torch.softmax(-d/float(model.T),dim=1)\n        eff=(1/(w.square().sum(1)+1e-12)).cpu().numpy(); avd=(w*d).sum(1).cpu().numpy(); k=min(32,Zs.shape[0]); ix=d.topk(k,largest=False).indices\n        topy=yt[ix].cpu().numpy(); same=(w* (yt[None,:]==torch.as_tensor(yv[s:s+len(q)],device=DEVICE)[:,None]).float()).sum(1).cpu().numpy()\n        for j in range(len(q)): rows.append((int(yv[s+j]),float(eff[j]),float(avd[j]),float(same[j]),float(np.mean(topy[j]==1)),float(np.mean(topy[j]==2)),float(np.mean(topy[j]==3)),float(np.mean(topy[j]==4))))\n    z=pd.DataFrame(rows,columns=[\'y\',\'effective_neighbors\',\'weighted_distance\',\'same_target_mass\',\'top32_p1\',\'top32_p2\',\'top32_p3\',\'top32_p4\'])\n    return z\n\ntrmask=TRAIN[\'__fold__\'].to_numpy()!=FOLD; vamask=~trmask\nOUTER_TR=TRAIN.loc[trmask].reset_index(drop=True); OUTER_VA=TRAIN.loc[vamask].reset_index(drop=True); ytr=OUTER_TR[\'life_satisfaction\'].to_numpy(float); yva=OUTER_VA[\'life_satisfaction\'].to_numpy(float)\n\ntry:\n    if MODE==\'runtime\':\n        prep=Prep().fit(OUTER_TR); Xtr=prep.transform(OUTER_TR); Xv=prep.transform(OUTER_VA.iloc[:min(2048,len(OUTER_VA))]); PREP_NNUM=prep.n_num\n        cfg=json.load(open(ROOT/\'official_default_v12.json\'))[\'modernNCA\']; t=time.time(); model,mean,std,Xs,Ys=train_fixed(Xtr,ytr,cfg,42,1); pv=pred(model,Xv,Xs,Ys,mean,std)\n        rec={\'status\':\'success\',\'fold\':FOLD,\'train_rows\':len(Xtr),\'probe_rows\':len(Xv),\'features\':Xtr.shape[1],\'n_num\':PREP_NNUM,\'seconds\':time.time()-t,\'finite\':bool(np.isfinite(pv).all()),\'peak_cuda\':int(torch.cuda.max_memory_allocated()),\'prep\':prep.manifest(),\'talent_commit\':CFG[\'talent_commit\'],\'modernnca_model_sha256\':CFG[\'modernnca_model_sha256\'],\'config_manifest_sha256\':CFG[\'config_manifest_sha256\']}; json.dump(rec,open(OUT/\'runtime_proof.json\',\'w\'),indent=2); print(json.dumps(rec,indent=2))\n    elif MODE in (\'screen\',\'full\'):\n        # Inner HPO: one fixed stratified split inside outer train, seed 42 only.\n        sss=StratifiedShuffleSplit(n_splits=1,test_size=.2,random_state=20260908+FOLD); ii,iv=next(sss.split(np.zeros(len(OUTER_TR)),ytr.astype(int)))\n        ITR=OUTER_TR.iloc[ii].reset_index(drop=True); IVA=OUTER_TR.iloc[iv].reset_index(drop=True); yi=ytr[ii]; yiv=ytr[iv]\n        prep_i=Prep().fit(ITR); Xi=prep_i.transform(ITR); Xiv=prep_i.transform(IVA); PREP_NNUM=prep_i.n_num\n        hrows=[]; bestrec=None\n        for cfg in CONFIGS:\n            t=time.time(); model,rm,ep,mean,std,hist,Xs,Ys=train_early(Xi,yi,Xiv,yiv,cfg,42)\n            rec={\'config_id\':cfg[\'config_id\'],\'rmse\':rm,\'best_epoch\':ep,\'seconds\':time.time()-t,\'peak_cuda\':int(torch.cuda.max_memory_allocated())}; hrows.append(rec); pd.DataFrame(hrows).to_csv(OUT/\'hpo_metrics.csv\',index=False)\n            json.dump(hist,open(OUT/f\'history_config{cfg["config_id"]}.json\',\'w\'))\n            if bestrec is None or (rm, cfg[\'config_id\']) < (bestrec[\'rmse\'],bestrec[\'config_id\']): bestrec=rec\n            del model,Xs,Ys; gc.collect(); torch.cuda.empty_cache()\n        if len(hrows)!=8 or sorted(int(r[\'config_id\']) for r in hrows)!=list(range(8)):\n            raise RuntimeError(f\'Fixed HPO incomplete: completed {[r["config_id"] for r in hrows]}\')\n        sel=CONFIGS[int(bestrec[\'config_id\'])]; epochs=int(bestrec[\'best_epoch\'])+1\n        # Full outer-training refit, two fixed seeds, average outer predictions.\n        prep=Prep().fit(OUTER_TR); Xtr=prep.transform(OUTER_TR); Xv=prep.transform(OUTER_VA); PREP_NNUM=prep.n_num\n        preds=[]; states=[]; geom=None\n        for seed in (42,2026):\n            model,mean,std,Xs,Ys=train_fixed(Xtr,ytr,sel,seed,epochs); p=pred(model,Xv,Xs,Ys,mean,std); preds.append(p)\n            torch.save({\'state\':model.state_dict(),\'mean\':mean,\'std\':std,\'config_id\':int(bestrec[\'config_id\']),\'epochs\':epochs,\'n_num\':PREP_NNUM},OUT/f\'model_seed{seed}.pt\')\n            if seed==42:\n                del Xs,Ys; torch.cuda.empty_cache(); geom=geometry(model,Xv,Xtr,yva,ytr); geom.to_csv(OUT/\'geometry_rows.csv\',index=False)\n            else:\n                del Xs,Ys\n            del model; gc.collect(); torch.cuda.empty_cache()\n        po=np.mean(np.vstack(preds),axis=0)\n        if not np.isfinite(po).all(): raise FloatingPointError(\'Non-finite outer predictions\')\n        np.savez_compressed(OUT/\'outer_predictions.npz\',id=OUTER_VA.id.to_numpy(),y=yva,pred=po,fold=np.full(len(po),FOLD))\n        meta={\'status\':\'success\',\'fold\':FOLD,\'selected_config_id\':int(bestrec[\'config_id\']),\'selected_inner_rmse\':float(bestrec[\'rmse\']),\'selected_best_epoch\':int(bestrec[\'best_epoch\']),\'refit_epochs\':epochs,\'outer_rows\':len(OUTER_TR),\'valid_rows\':len(OUTER_VA),\'features\':Xtr.shape[1],\'n_num\':PREP_NNUM,\'prep\':prep.manifest(),\'talent_commit\':CFG[\'talent_commit\'],\'modernnca_model_sha256\':CFG[\'modernnca_model_sha256\'],\'config_manifest_sha256\':CFG[\'config_manifest_sha256\']}; json.dump(meta,open(OUT/\'fold_summary.json\',\'w\'),indent=2)\n        if MODE==\'full\':\n            Xt=prep.transform(TEST); tp=[]\n            for seed in (42,2026):\n                ck=torch.load(OUT/f\'model_seed{seed}.pt\',map_location=DEVICE); model=build_model(Xtr.shape[1],PREP_NNUM,sel); model.load_state_dict(ck[\'state\']); Xs=torch.as_tensor(Xtr,device=DEVICE,dtype=torch.float32); Ys=torch.as_tensor((ytr-ck[\'mean\'])/ck[\'std\'],device=DEVICE,dtype=torch.float32); tp.append(pred(model,Xt,Xs,Ys,ck[\'mean\'],ck[\'std\'])); del model,Xs,Ys; torch.cuda.empty_cache()\n            _tp=np.mean(np.vstack(tp),axis=0)\n            if not np.isfinite(_tp).all(): raise FloatingPointError(\'Non-finite full-fold test predictions\')\n            np.savez_compressed(OUT/\'test_predictions.npz\',id=TEST.id.to_numpy(),pred=_tp,fold=np.full(len(TEST),FOLD))\n        print(json.dumps(meta,indent=2))\n    elif MODE==\'resume_test\':\n        src=ROOT/f\'screen_fold{FOLD}\'; meta=json.load(open(src/\'fold_summary.json\')); sel=CONFIGS[int(meta[\'selected_config_id\'])]\n        prep=Prep().fit(OUTER_TR); Xtr=prep.transform(OUTER_TR); Xt=prep.transform(TEST); PREP_NNUM=prep.n_num; tp=[]\n        for seed in (42,2026):\n            ck=torch.load(src/f\'model_seed{seed}.pt\',map_location=DEVICE); model=build_model(Xtr.shape[1],PREP_NNUM,sel); model.load_state_dict(ck[\'state\']); Xs=torch.as_tensor(Xtr,device=DEVICE,dtype=torch.float32); Ys=torch.as_tensor((ytr-ck[\'mean\'])/ck[\'std\'],device=DEVICE,dtype=torch.float32); tp.append(pred(model,Xt,Xs,Ys,ck[\'mean\'],ck[\'std\'])); del model,Xs,Ys; torch.cuda.empty_cache()\n        _tp=np.mean(np.vstack(tp),axis=0)\n        if not np.isfinite(_tp).all(): raise FloatingPointError(\'Non-finite resume-test predictions\')\n        np.savez_compressed(OUT/\'test_predictions.npz\',id=TEST.id.to_numpy(),pred=_tp,fold=np.full(len(TEST),FOLD)); json.dump({\'status\':\'success\',\'fold\':FOLD},open(OUT/\'resume_summary.json\',\'w\'))\nexcept Exception as e:\n    fail={\'status\':\'failed\',\'mode\':MODE,\'fold\':FOLD,\'error_type\':type(e).__name__,\'error\':str(e),\'traceback\':traceback.format_exc(),\'cuda_allocated\':int(torch.cuda.memory_allocated()) if torch.cuda.is_available() else None,\'cuda_reserved\':int(torch.cuda.memory_reserved()) if torch.cuda.is_available() else None,\'cuda_peak\':int(torch.cuda.max_memory_allocated()) if torch.cuda.is_available() else None}\n    json.dump(fail,open(OUT/\'failure.json\',\'w\'),indent=2); print(json.dumps(fail,indent=2),file=sys.stderr); raise\n'
worker_path=OUT/'modernnca_worker.py'; worker_path.write_text(WORKER_SOURCE); compile(WORKER_SOURCE,str(worker_path),'exec'); print('Worker written',worker_path,'bytes',worker_path.stat().st_size)


Worker written /kaggle/working/oqdq5_v12/modernnca_worker.py bytes 19041


In [7]:
runtime_contract=asdict(CFG)
runtime_contract.update({
    'talent_commit':TALENT_COMMIT,
    'modernnca_model_sha256':actual_sha,
    'modernnca_method_sha256':method_sha,
    'config_manifest_sha256':config_sha,
})
with open(OUT/'runtime_contract.json','w') as f: json.dump(runtime_contract,f,indent=2)
with open(OUT/'official_default_v12.json','w') as f: json.dump(official_default,f,indent=2)
with open(OUT/'provenance_preflight.json','w') as f:
    json.dump({
        'TALENT_version':getattr(TALENT,'__version__','unknown'),
        'TALENT_commit':TALENT_COMMIT,
        'TALENT_direct_url':DIRECT_URL,
        'modernNCA_source_sha256':actual_sha,
        'modernNCA_method_sha256':method_sha,
        'config_manifest_sha256':config_sha,
        'official_model_file':str(model_file),
        'official_method_file':str(method_file),
        'dependency_report':req_report,
        'data_dir':str(DATA),
        'historical_source':STORE.source(),
        'torch':torch.__version__,
        'python':platform.python_version(),
    },f,indent=2)

def _show_failure(mode,fold,log):
    txt=log.read_text(errors='replace') if log.exists() else ''
    print(f'--- {mode} fold {fold} underlying log ---\n{txt[-20000:]}')
    fp=OUT/f'{mode}_fold{fold}'/'failure.json'
    if fp.exists():
        print('--- failure manifest ---')
        print(fp.read_text(errors='replace')[-20000:])

def run_worker(mode,fold,gpu,wait=True):
    d=OUT/f'{mode}_fold{fold}'
    if d.exists(): shutil.rmtree(d)
    log=OUT/f'{mode}_fold{fold}.log'
    if log.exists(): log.unlink()
    env=os.environ.copy()
    env['CUDA_VISIBLE_DEVICES']=str(gpu)
    env['PYTHONUNBUFFERED']='1'
    fh=open(log,'w')
    p=subprocess.Popen(
        [sys.executable,str(OUT/'modernnca_worker.py'),'--root',str(OUT),'--mode',mode,'--fold',str(fold)],
        env=env,stdout=fh,stderr=subprocess.STDOUT
    )
    if not wait:
        return p,fh,log
    rc=p.wait(); fh.close()
    if rc!=0:
        _show_failure(mode,fold,log)
        raise RuntimeError(f'{mode} worker fold {fold} failed rc={rc}')
    return d

def wait_parallel(jobs,stage):
    # Wait for isolated GPU workers; terminate every still-running sibling immediately if one fails.
    active=list(jobs)
    completed=[]
    while active:
        failure=None
        next_active=[]
        for item in active:
            fold,p,fh,log,mode=item
            rc=p.poll()
            if rc is None:
                next_active.append(item)
            else:
                if not fh.closed: fh.close()
                if rc!=0 and failure is None:
                    failure=(fold,rc,log,mode)
                elif rc==0:
                    completed.append(fold)
        if failure is not None:
            # Stop all jobs from this batch that are still alive, regardless of list position.
            for fold2,p2,fh2,log2,mode2 in active:
                if p2.poll() is None:
                    p2.terminate()
            for fold2,p2,fh2,log2,mode2 in active:
                if p2.poll() is None:
                    try: p2.wait(timeout=20)
                    except subprocess.TimeoutExpired:
                        p2.kill(); p2.wait()
                if not fh2.closed: fh2.close()
            fold,rc,log,mode=failure
            _show_failure(mode,fold,log)
            raise RuntimeError(f'{stage}: {mode} fold {fold} failed rc={rc}; sibling workers terminated.')
        active=next_active
        if active: time.sleep(1)
    return completed


## 4. Stage R — realistic runtime proof

In [8]:
run_worker('runtime',0,0)
rproof=json.load(open(OUT/'runtime_fold0'/'runtime_proof.json'))
assert rproof['status']=='success' and rproof['finite'] and rproof['train_rows']>70000 and rproof['features']>100
print(json.dumps(rproof,indent=2))

{
  "status": "success",
  "fold": 0,
  "train_rows": 75730,
  "probe_rows": 2048,
  "features": 1125,
  "n_num": 19,
  "seconds": 14.063180208206177,
  "finite": true,
  "peak_cuda": 2613769216,
  "prep": {
    "n_num": 19,
    "ohe_dim": 1094,
    "total_nominal_categories": {
      "skipped_prescription_due_to_cost": 7,
      "works_in_healthcare": 7,
      "has_regular_care_provider": 8,
      "has_disability_income": 7,
      "difficulty_social_activities": 8,
      "geographic_region": 6,
      "has_lowincome_gov_insurance": 6,
      "has_wage_income": 7,
      "ever_smoked_cigar": 7,
      "flu_vaccine_month": 17,
      "has_indigenous_insurance": 6,
      "takes_depression_medication": 7,
      "volunteers_in_healthcare": 7,
      "virtual_medical_appointment": 7,
      "employer_offers_insurance": 7,
      "pseudo_psu": 112,
      "indigenous_health_service": 5,
      "lung_disease_ever": 5,
      "ever_served_military": 6,
      "dental_only_plan": 5,
      "policyholder_plan

## 5. Stage S — full nested screen on frozen folds 0 and 2, one process per T4

In [9]:
jobs=[]
for fold,gpu in zip(CFG.screen_folds,(0,1)):
    p,fh,log=run_worker('screen',fold,gpu,wait=False)
    jobs.append((fold,p,fh,log,'screen'))
done=wait_parallel(jobs,'Stage S')
assert sorted(done)==sorted(CFG.screen_folds)
print('Stage S workers completed:',done)


Stage S workers completed: [0, 2]


In [10]:
def qwk(a,b): return float(cohen_kappa_score(a,b,weights='quadratic'))
def disc(s,t): t=np.asarray(t,float); return (1+(s>=t[0])+(s>=t[1])+(s>=t[2])).astype(int)
def fit_thr(yy,ss,start=(1.57,2.01,2.51)):
    def obj(t):
        t=np.sort(np.asarray(t,float))
        if min(t[1]-t[0],t[2]-t[1])<.02:return 10
        return -qwk(yy,disc(ss,t))
    r=minimize(obj,np.array(start),method='Powell',bounds=((1.15,1.95),(1.55,2.45),(2.05,3.25)),options={'maxiter':250,'xtol':1e-5,'ftol':1e-8}); return np.sort(r.x)
def auc12(yy,ss): m=np.isin(yy,[1,2]); return float(roc_auc_score((yy[m]==2).astype(int),ss[m]))
def bauc(yy,ss,k): return float(roc_auc_score((yy>k).astype(int),ss))

screen={}
for f in CFG.screen_folds:
    z=np.load(OUT/f'screen_fold{f}'/'outer_predictions.npz'); ids=z['id']; idx=pd.Index(train_ids).get_indexer(ids); assert (idx>=0).all() and np.all(folds[idx]==f)
    screen[f]={'idx':idx,'y':z['y'].astype(int),'pred':z['pred'].astype(float),'rec':recovery_oof[idx],'v2':v2_oof[idx].astype(float)}
rows=[]; all_y=[];all_p=[];all_r=[]
for f,other in [(0,2),(2,0)]:
    d=screen[f]; o=screen[other]; tn=fit_thr(o['y'],o['pred']); tr=fit_thr(o['y'],o['rec']); tv=fit_thr(o['y'],o['v2']); cn=disc(d['pred'],tn); cr=disc(d['rec'],tr); cv=disc(d['v2'],tv)
    row={'fold':f,'n':len(d['y']),'nca_qwk':qwk(d['y'],cn),'recovery_qwk':qwk(d['y'],cr),'delta_qwk':qwk(d['y'],cn)-qwk(d['y'],cr),'v2_qwk':qwk(d['y'],cv),'delta_qwk_vs_v2':qwk(d['y'],cn)-qwk(d['y'],cv),'class_disagreement_rate':float(np.mean(cn!=cr)),'nca_auc_1v2':auc12(d['y'],d['pred']),'recovery_auc_1v2':auc12(d['y'],d['rec']),'delta_auc_1v2':auc12(d['y'],d['pred'])-auc12(d['y'],d['rec']),'nca_auc_ygt1':bauc(d['y'],d['pred'],1),'recovery_auc_ygt1':bauc(d['y'],d['rec'],1),'delta_auc_ygt1':bauc(d['y'],d['pred'],1)-bauc(d['y'],d['rec'],1),'nca_auc_ygt2':bauc(d['y'],d['pred'],2),'recovery_auc_ygt2':bauc(d['y'],d['rec'],2),'delta_auc_ygt2':bauc(d['y'],d['pred'],2)-bauc(d['y'],d['rec'],2),'nca_auc_ygt3':bauc(d['y'],d['pred'],3),'recovery_auc_ygt3':bauc(d['y'],d['rec'],3),'delta_auc_ygt3':bauc(d['y'],d['pred'],3)-bauc(d['y'],d['rec'],3),'v2_auc_1v2':auc12(d['y'],d['v2']),'v2_auc_ygt1':bauc(d['y'],d['v2'],1),'nca_rmse':float(np.sqrt(mean_squared_error(d['y'],d['pred']))),'recovery_rmse':float(np.sqrt(mean_squared_error(d['y'],d['rec']))),'nca_mae':float(mean_absolute_error(d['y'],d['pred'])),'recovery_mae':float(mean_absolute_error(d['y'],d['rec'])),'nca_spearman':float(spearmanr(d['y'],d['pred'])[0]),'recovery_spearman':float(spearmanr(d['y'],d['rec'])[0]),'threshold_source_fold':other}
    rows.append(row); all_y.append(d['y']);all_p.append(d['pred']);all_r.append(d['rec'])
sdf=pd.DataFrame(rows); sdf.to_csv(OUT/'stage_s_metrics.csv',index=False)
y2=np.concatenate(all_y);p2=np.concatenate(all_p);r2=np.concatenate(all_r)
mean_a12=float(sdf.delta_auc_1v2.mean()); mean_ay1=float(sdf.delta_auc_ygt1.mean()); mean_q=float(sdf.delta_qwk.mean()); rcorr=float(np.corrcoef(p2-y2,r2-y2)[0,1])
no_cat=bool((sdf[['delta_auc_1v2','delta_auc_ygt1']].to_numpy()>=CFG.stage_catastrophic_auc).all())
gateA=bool(no_cat and (mean_a12>=CFG.stage_auc_gain or mean_ay1>=CFG.stage_auc_gain))
gateB=bool(mean_q>=CFG.stage_qwk_gain)
novel_auc_ok=bool((sdf[['delta_auc_1v2','delta_auc_ygt1']].to_numpy()>=CFG.stage_novel_auc_floor).all())
gateC=bool(mean_q>=CFG.stage_novel_qwk_floor and rcorr<CFG.stage_novel_resid_corr and novel_auc_ok)
STAGE_PROMOTED=bool(gateA or gateB or gateC)
delta=p2-r2
ss={'mean_delta_auc_1v2':mean_a12,'mean_delta_auc_ygt1':mean_ay1,'mean_delta_qwk':mean_q,'residual_corr':rcorr,'prediction_corr':float(np.corrcoef(p2,r2)[0,1]),'score_delta_quantiles':list(map(float,np.quantile(delta,[0,.01,.05,.25,.5,.75,.95,.99,1]))),'mean_abs_score_delta':float(np.mean(np.abs(delta))),'no_catastrophic_auc_regression':no_cat,'ranking_gate':gateA,'qwk_gate':gateB,'novelty_critical_auc_ok':novel_auc_ok,'novelty_gate':gateC,'promoted':STAGE_PROMOTED}
json.dump(ss,open(OUT/'stage_s_summary.json','w'),indent=2); print(sdf.to_string(index=False)); print(json.dumps(ss,indent=2))

 fold     n  nca_qwk  recovery_qwk  delta_qwk   v2_qwk  delta_qwk_vs_v2  class_disagreement_rate  nca_auc_1v2  recovery_auc_1v2  delta_auc_1v2  nca_auc_ygt1  recovery_auc_ygt1  delta_auc_ygt1  nca_auc_ygt2  recovery_auc_ygt2  delta_auc_ygt2  nca_auc_ygt3  recovery_auc_ygt3  delta_auc_ygt3  v2_auc_1v2  v2_auc_ygt1  nca_rmse  recovery_rmse  nca_mae  recovery_mae  nca_spearman  recovery_spearman  threshold_source_fold
    0 18933 0.481007      0.490727  -0.009720 0.490678        -0.009671                 0.097502     0.762325          0.767312      -0.004988      0.777212           0.782070       -0.004858      0.879073           0.883003       -0.003930      0.833725           0.833190        0.000535    0.766944     0.781705  0.515558       0.512201 0.429381      0.426482      0.507131           0.515551                      2
    2 18933 0.465658      0.476416  -0.010758 0.475891        -0.010234                 0.101727     0.759263          0.762950      -0.003687      0.773244      

## 6. Full five-fold validation only if Stage S promotes

In [11]:
FULL_AVAILABLE=False
if STAGE_PROMOTED:
    # Reuse Stage-S OOF/models for folds 0/2. Train only missing folds, max two isolated workers.
    for batch in [[(1,0),(3,1)],[(4,0)]]:
        jobs=[]
        for f,g in batch:
            p,fh,log=run_worker('full',f,g,wait=False)
            jobs.append((f,p,fh,log,'full'))
        done=wait_parallel(jobs,'Full CV')
        if sorted(done)!=sorted(f for f,_ in batch):
            raise RuntimeError(f'Full-CV batch incomplete: expected {batch}, completed {done}')
    # Test inference for already-trained Stage-S fold 0/2 models, without retraining.
    jobs=[]
    for f,g in zip((0,2),(0,1)):
        p,fh,log=run_worker('resume_test',f,g,wait=False)
        jobs.append((f,p,fh,log,'resume_test'))
    done=wait_parallel(jobs,'Resume test')
    if sorted(done)!=[0,2]:
        raise RuntimeError(f'Resume-test incomplete: {done}')
    FULL_AVAILABLE=True
else:
    print('V12 falsified at Stage S. Full CV/test inference intentionally skipped; no submission can be produced.')


V12 falsified at Stage S. Full CV/test inference intentionally skipped; no submission can be produced.


In [12]:
final_summary={'stage_s':ss,'full_available':FULL_AVAILABLE,'recommended_candidate':None,'submission_path':None}
if FULL_AVAILABLE:
    nca_oof=np.full(len(y),np.nan,dtype=float); coverage=np.zeros(len(y),dtype=np.int8); nca_test_folds=[]
    id_index=pd.Index(train_ids)
    for f in range(5):
        src=OUT/(f'screen_fold{f}' if f in CFG.screen_folds else f'full_fold{f}')
        z=np.load(src/'outer_predictions.npz',allow_pickle=False)
        zid=z['id']; zp=np.asarray(z['pred'],float)
        idx=id_index.get_indexer(zid)
        if (idx<0).any() or len(np.unique(zid))!=len(zid) or not np.all(folds[idx]==f):
            raise RuntimeError(f'Invalid outer prediction ID/fold contract for fold {f}')
        if not np.isfinite(zp).all() or len(zp)!=len(idx):
            raise RuntimeError(f'Invalid outer prediction values for fold {f}')
        nca_oof[idx]=zp; coverage[idx]+=1
        tsrc=OUT/(f'resume_test_fold{f}' if f in CFG.screen_folds else f'full_fold{f}')
        tz=np.load(tsrc/'test_predictions.npz',allow_pickle=False)
        if not np.array_equal(tz['id'],test_ids):
            raise RuntimeError(f'Test ID alignment failure for fold {f}')
        tp=np.asarray(tz['pred'],float)
        if len(tp)!=len(test_ids) or not np.isfinite(tp).all():
            raise RuntimeError(f'Invalid test predictions for fold {f}')
        nca_test_folds.append(tp)
    if not np.all(coverage==1) or not np.isfinite(nca_oof).all():
        vals,cnt=np.unique(coverage,return_counts=True)
        raise RuntimeError(f'OOF coverage invalid: {dict(zip(vals.tolist(),cnt.tolist()))}')
    nca_test=np.mean(np.vstack(nca_test_folds),axis=0)
    if not np.isfinite(nca_test).all():
        raise RuntimeError('Non-finite averaged V12 test predictions')
    # Cross-fitted standalone thresholds.
    nca_cls=np.zeros(len(y),dtype=int); thr=[]
    for f in range(5):
        tr=folds!=f; va=~tr; t=fit_thr(y[tr],nca_oof[tr]); nca_cls[va]=disc(nca_oof[va],t); thr.append(t)
    if not set(np.unique(nca_cls)).issubset({1,2,3,4}) or (nca_cls==0).any(): raise RuntimeError('Standalone cross-fit calibration coverage failure')
    # Frozen RMSE-selected blend alpha, nested by outer fold.
    alphas=np.array([0,.25,.5,.75,1.]); blend=np.full(len(y),np.nan,dtype=float); alpha_rows=[]
    for f in range(5):
        tr=folds!=f; va=~tr; rms=[np.sqrt(mean_squared_error(y[tr],(1-a)*recovery_oof[tr]+a*nca_oof[tr])) for a in alphas]; a=float(alphas[int(np.argmin(rms))]); blend[va]=(1-a)*recovery_oof[va]+a*nca_oof[va]; alpha_rows.append({'fold':f,'alpha':a,'train_rmse':float(min(rms))})
    if not np.isfinite(blend).all(): raise RuntimeError('Blend outer-fold coverage failure')
    blend_cls=np.zeros(len(y),dtype=int); bthr=[]
    for f in range(5): tr=folds!=f; va=~tr; t=fit_thr(y[tr],blend[tr]); blend_cls[va]=disc(blend[va],t); bthr.append(t)
    if not set(np.unique(blend_cls)).issubset({1,2,3,4}) or (blend_cls==0).any(): raise RuntimeError('Blend cross-fit calibration coverage failure')
    pd.DataFrame(alpha_rows).to_csv(OUT/'blend_fold_alphas.csv',index=False)
    fold_rows=[]
    for f in range(5):
        m=folds==f
        fold_rows.append({'fold':f,'n':int(m.sum()),'recovery_qwk':qwk(y[m],recovery_class[m]),'modernnca_qwk':qwk(y[m],nca_cls[m]),'modernnca_delta_qwk':qwk(y[m],nca_cls[m])-qwk(y[m],recovery_class[m]),'blend_qwk':qwk(y[m],blend_cls[m]),'blend_delta_qwk':qwk(y[m],blend_cls[m])-qwk(y[m],recovery_class[m]),'recovery_auc_1v2':auc12(y[m],recovery_oof[m]),'modernnca_auc_1v2':auc12(y[m],nca_oof[m]),'blend_auc_1v2':auc12(y[m],blend[m]),'recovery_auc_ygt1':bauc(y[m],recovery_oof[m],1),'modernnca_auc_ygt1':bauc(y[m],nca_oof[m],1),'blend_auc_ygt1':bauc(y[m],blend[m],1)})
    pd.DataFrame(fold_rows).to_csv(OUT/'full_fold_metrics.csv',index=False)

    def evaluate(name,score,cls):
        fq=[]
        for f in range(5):
            m=folds==f; fq.append(qwk(y[m],cls[m])-qwk(y[m],recovery_class[m]))
        rng=np.random.default_rng(SEED+1200); boot=np.empty(CFG.bootstrap_reps)
        for b in range(CFG.bootstrap_reps):
            ix=rng.integers(0,len(y),len(y)); boot[b]=qwk(y[ix],cls[ix])-qwk(y[ix],recovery_class[ix])
        np.save(OUT/f'bootstrap_{name}_v12.npy',boot)
        sane=bool(np.isfinite(score).all() and np.std(score)>1e-6 and np.max(np.abs(score))<100 and set(np.unique(cls)).issubset({1,2,3,4}))
        return {'name':name,'qwk':qwk(y,cls),'recovery_qwk':qwk(y,recovery_class),'delta_qwk':qwk(y,cls)-qwk(y,recovery_class),'auc_1v2':auc12(y,score),'recovery_auc_1v2':auc12(y,recovery_oof),'delta_auc_1v2':auc12(y,score)-auc12(y,recovery_oof),'auc_ygt1':bauc(y,score,1),'delta_auc_ygt1':bauc(y,score,1)-bauc(y,recovery_oof,1),'rmse':float(np.sqrt(mean_squared_error(y,score))),'mae':float(mean_absolute_error(y,score)),'spearman':float(spearmanr(y,score)[0]),'prediction_corr':float(np.corrcoef(score,recovery_oof)[0,1]),'residual_corr':float(np.corrcoef(score-y,recovery_oof-y)[0,1]),'fold_deltas':list(map(float,fq)),'positive_folds':int(np.sum(np.array(fq)>=0)),'min_fold_delta':float(min(fq)),'bootstrap_p_gt0':float(np.mean(boot>0)),'bootstrap_median':float(np.median(boot)),'bootstrap_95':list(map(float,np.quantile(boot,[.025,.975]))),'sane_prediction':sane,'global_calibration_used':False}
    E1=evaluate('modernnca',nca_oof,nca_cls); E2=evaluate('blend',blend,blend_cls); evals=[E1,E2]
    pd.DataFrame([{k:v for k,v in e.items() if not isinstance(v,list)} for e in evals]).to_csv(OUT/'full_candidate_metrics.csv',index=False)
    # Candidate can be selected only among the two preregistered pipelines.
    best=max(evals,key=lambda e:(e['qwk'],e['delta_auc_1v2']))
    scientific=bool(best['delta_qwk']>=CFG.scientific_qwk_gain and (best['positive_folds']>=4 or best['bootstrap_p_gt0']>=.95) and best['delta_auc_1v2']>=0 and best['delta_auc_ygt1']>=0)
    submit=bool(best['delta_qwk']>=CFG.submission_qwk_gain and best['positive_folds']>=4 and best['min_fold_delta']>=CFG.submission_min_fold and best['bootstrap_p_gt0']>=CFG.submission_boot_p and best['delta_auc_1v2']>=0 and best['sane_prediction'] and not best['global_calibration_used'])
    challenge=bool(submit and (best['delta_qwk']>=CFG.challenge_qwk_gain or best['delta_auc_1v2']>=CFG.challenge_auc_gain))
    breakthrough=bool(best['delta_qwk']>=0.005 or best['delta_auc_1v2']>=0.010)
    # Save OOF and diagnostics regardless of submission gate.
    pd.DataFrame({'id':train_ids,'life_satisfaction':y,'fold':folds,'recovery_score':recovery_oof,'recovery_class':recovery_class,'modernnca_score':nca_oof,'modernnca_class':nca_cls,'blend_score':blend,'blend_class':blend_cls}).to_csv(OUT/'oof_v12.csv',index=False)
    np.savez_compressed(OUT/'test_scores_v12.npz',id=test_ids,modernnca=nca_test,recovery=recovery_test)
    # Geometry summaries are diagnostic only.
    geos=[]
    for f in range(5):
        src=OUT/(f'screen_fold{f}' if f in CFG.screen_folds else f'full_fold{f}')/'geometry_rows.csv'
        g=pd.read_csv(src); g['fold']=f; geos.append(g)
    G=pd.concat(geos,ignore_index=True); G.groupby('y').mean(numeric_only=True).to_csv(OUT/'geometry_summary_by_class.csv')
    submission_path=None
    if submit:
        if best['name']=='modernnca': test_score=nca_test; final_t=np.median(np.vstack(thr),axis=0); final_alpha=1.0
        else:
            final_alpha=float(np.median([r['alpha'] for r in alpha_rows])); test_score=(1-final_alpha)*recovery_test+final_alpha*nca_test; final_t=np.median(np.vstack(bthr),axis=0)
        test_cls=disc(test_score,final_t); sub=pd.DataFrame({'id':test_ids,'life_satisfaction':test_cls.astype(int)})
        assert len(sub)==CFG.expected_test_ids and sub.id.nunique()==CFG.expected_test_ids and set(sub.life_satisfaction).issubset({1,2,3,4})
        submission_path=OUT/'submission_v12.csv'; sub.to_csv(submission_path,index=False)
        pd.DataFrame({'id':test_ids,'modernnca_score':nca_test,'recovery_score':recovery_test,'final_score':test_score,'final_class':test_cls}).to_csv(OUT/'test_predictions_v12.csv',index=False)
    final_summary.update({'full_available':True,'standalone':E1,'blend':E2,'recommended_candidate':best['name'],'scientific_improvement':scientific,'submission_gate':submit,'challenge_gate':challenge,'breakthrough_criterion':breakthrough,'submission_path':None if submission_path is None else str(submission_path),'final_alpha':None if not submit else final_alpha})
    print(json.dumps(final_summary,indent=2))
else:
    print(json.dumps(final_summary,indent=2))

{
  "stage_s": {
    "mean_delta_auc_1v2": -0.0043373497048324605,
    "mean_delta_auc_ygt1": -0.004237685067410957,
    "mean_delta_qwk": -0.010239081900547342,
    "residual_corr": 0.9923181243084308,
    "prediction_corr": 0.9794428530660368,
    "score_delta_quantiles": [
      -0.7285690102070213,
      -0.17560840416036697,
      -0.10051037018680153,
      -0.03453197626179938,
      0.003122395180123916,
      0.03857834129757087,
      0.09744482405025473,
      0.15830599424468086,
      0.5708660702097461
    ],
    "mean_abs_score_delta": 0.047332905446649705,
    "no_catastrophic_auc_regression": true,
    "ranking_gate": false,
    "qwk_gate": false,
    "novelty_critical_auc_ok": false,
    "novelty_gate": false,
    "promoted": false
  },
  "full_available": false,
  "recommended_candidate": null,
  "submission_path": null
}


## 7. Final artifact/provenance summary

In [13]:
def sha(path):
    h=hashlib.sha256();
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1<<20),b''): h.update(b)
    return h.hexdigest()
final_summary['version']='v12_modernnca_learned_geometry_full_target_ordinal_regression'
final_summary['frozen_recovery_public_lb']=CFG.recovery_lb
final_summary['gates']={'stage_s':{'ranking_gain':CFG.stage_auc_gain,'qwk_gain':CFG.stage_qwk_gain,'novelty_qwk_floor':CFG.stage_novel_qwk_floor,'novelty_residual_corr_lt':CFG.stage_novel_resid_corr},'scientific_qwk_gain':CFG.scientific_qwk_gain,'submission_qwk_gain':CFG.submission_qwk_gain,'submission_boot_p':CFG.submission_boot_p,'challenge_qwk_gain':CFG.challenge_qwk_gain,'challenge_auc_gain':CFG.challenge_auc_gain}
final_summary['provenance']={'TALENT_version':getattr(TALENT,'__version__','unknown'),'TALENT_commit':TALENT_COMMIT,'modernNCA_source_sha256':actual_sha,'modernNCA_method_sha256':method_sha,'config_manifest_sha256':config_sha,'dependency_requirements':req_report,'data_dir':str(DATA),'historical_source':STORE.source(),'train_seeds':list(CFG.train_seeds),'HPO_seed':HPO_SEED,'base_models_retrained':False,'official_architecture_import':'TALENT.model.models.modernNCA.ModernNCA'}
if final_summary.get('submission_path'): final_summary['submission_sha256']=sha(final_summary['submission_path'])
json.dump(final_summary,open(OUT/'v12_summary.json','w'),indent=2)
STORE.close()
print('FINAL STATUS:', 'SUBMISSION READY' if final_summary.get('submission_path') else ('FULL CV, NOT SUBMITTABLE' if FULL_AVAILABLE else 'FALSIFIED AT STAGE S'))
print('Summary:',OUT/'v12_summary.json')

FINAL STATUS: FALSIFIED AT STAGE S
Summary: /kaggle/working/oqdq5_v12/v12_summary.json
